# synthesis

information loss against cost, across every experiment.

In [ ]:
from pathlib import Path

import pandas as pd

from causality_bench.provenance import read_csv

pd.set_option("display.width", 200)

# results/ sits beside notebooks/ in the repo root
RESULTS = Path.cwd().parent / "results"


def raw(name):
    return read_csv(RESULTS / "raw" / f"{name}.csv")


def processed(name):
    return read_csv(RESULTS / "processed" / f"{name}.csv")


the false-ordering rate and the concurrency ratio side by side, across every experiment.

In [ ]:
frames = []
for name in ["baseline", "scaling", "message_rate", "delay", "topology", "failure"]:
    frame = raw(name)
    frame["experiment"] = name
    frames.append(frame)

runs = pd.concat(frames, ignore_index=True)
runs["misrepresented_share"] = runs.lamport_false_orderings / runs.total_pairs
runs["vector_density"] = runs.mean_vector_nonzero_entries / runs.nodes
print("total runs:", len(runs))
runs.groupby("experiment")[["lamport_false_ordering_rate", "misrepresented_share", "vector_density"]].agg(
    ["min", "mean", "max"]
).round(4)

the trade-off in one table: the bytes per message a scalar clock saves, against the share of concurrent pairs it misorders to save them.

In [ ]:
cost = processed("scaling_summary")[["nodes", "vector_timestamp_bytes_mean"]].copy()
loss = processed("scaling_summary")[["nodes", "lamport_false_ordering_rate_mean"]]
trade = cost.merge(loss, on="nodes")
trade["bytes_per_message_saved"] = trade.vector_timestamp_bytes_mean - 8
trade